In [ ]:
import numpy as np
import open3d as o3d
from plyfile import PlyData 
from pycpd import DeformableRegistration
from scipy.spatial import cKDTree
import smplx
import torch
import trimesh


/Users/adeleyounis/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


0.19.0


In [3]:
sam3d_ply = o3d.io.read_triangle_mesh('/Users/adeleyounis/Downloads/point_cloud_sam3d.ply')

points = np.asarray(sam3d_ply.vertices)

def npy_to_o3dmesh(point_cloud, use_ball=False, visualize=False):

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(point_cloud)

    pcd.estimate_normals()

    if use_ball:
        # try with ball pivoting reconstruction, slower
        pcd.estimate_normals()

        dist = np.mean(pcd.compute_nearest_neighbor_distance())
        # radii_passes = [
        #     [dist*1.5, dist*2.5],
        #     [dist*3.5, dist*5.0],
        #     [dist*6.0, dist*9.0]
        # ]

        # meshes = []
        # for radii in radii_passes:
        #     m = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        #         pcd, o3d.utility.DoubleVector(radii)
        #     )
        #     meshes.append(m)

        # merged = meshes[0]
        # for m in meshes[1:]:
        #     merged += m

        # merged.remove_duplicated_vertices()
        # merged.remove_duplicated_triangles()
        # merged.remove_non_manifold_edges()
        # mesh = merged

        radii = [
            dist * 2.0,   # tight
            dist * 3.0,   # medium
            dist * 4.0    # slightly aggressive
        ]

        mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
            pcd,
            o3d.utility.DoubleVector(radii)
        )

        mesh.remove_non_manifold_edges()
        mesh.remove_degenerate_triangles()
        mesh.remove_unreferenced_vertices()

        mesh_t = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
        mesh_t = mesh_t.fill_holes(8000.0)
        mesh = mesh_t.to_legacy()

        # sample densely from your current mesh
        pcd2 = mesh.sample_points_poisson_disk(
            number_of_points=80000,
            init_factor=5
        )

        pcd2.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=0.03, max_nn=50)
        )

        pcd2.orient_normals_consistent_tangent_plane(100)

        mesh_p, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pcd2,
            depth=9
        )

        # crop to original bounds (removes floating junk)
        bbox = mesh.get_axis_aligned_bounding_box()
        mesh_p = mesh_p.crop(bbox)

        mesh_p.remove_non_manifold_edges()
        mesh_p.remove_degenerate_triangles()
        mesh_p.remove_unreferenced_vertices()
        print("Watertight:", mesh_p.is_watertight())

        triangles = np.asarray(mesh_p.triangles)
        vertices  = np.asarray(mesh_p.vertices)
        v0, v1, v2 = vertices[triangles[:, 0]], vertices[triangles[:, 1]], vertices[triangles[:, 2]] # vertexs of the triangle
        # einsum = Einstein sum: dot product between the cross product
        volume = np.abs(np.sum(np.einsum('ij,ij->i', v0, np.cross(v1 - v0, v2 - v0)))) / 6.0  # cross product for outward vector (flux)
        print("Volume:", volume)
    else:
        # try with poisson reconstruction, faster
        pcd.orient_normals_consistent_tangent_plane(100)

        mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pcd,
            depth=12,        # smoother = higher
            scale=1.0,
            linear_fit=True
        )

    mesh.compute_vertex_normals()
    
    if visualize:
        o3d.visualization.draw_geometries([mesh_p], mesh_show_back_face=True)
        # save as obj file 
        o3d.io.write_triangle_mesh("sam3d_reconstructed.obj", mesh_p)
    return pcd, mesh

pcd, mesh = npy_to_o3dmesh(points, use_ball=True, visualize=True)

[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
[Open3D WARNING] Ignoring attribute 'normals' for TensorMap with primary key 'indices'
Watertight: False
Volume: 0.030601754271925196
